[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/06_probabilistic_view_scratch_and_sklearn.ipynb)

# Bài 06 — Linear Regression theo template: xác suất, giả định, from scratch → thư viện, diễn giải

Bài này đi qua **template chuẩn cho mọi model** của khoá học, áp dụng cho Linear Regression.

| Phần | Linear Regression |
|---|---|
| Problem | dự đoán giá trị liên tục $y$ từ feature $\mathbf x$ |
| Intuition | tìm siêu phẳng "đi giữa" dữ liệu, tổng bình phương khoảng cách dọc nhỏ nhất |
| Geometry | $\hat{\mathbf y}$ là hình chiếu của $\mathbf y$ lên không gian cột của $X$ (bài 03) |
| Probability | MLE dưới giả định nhiễu Gauss (mục 1) |
| Objective | MSE: $\frac{1}{n}\lVert X\mathbf w + b - \mathbf y\rVert^2$ |
| Optimization | nghiệm đóng (normal equation / lstsq) hoặc gradient descent |
| Assumptions | mục 2 |
| Complexity | train $O(nd^2 + d^3)$ (lstsq) hoặc $O(knd)$ (GD); predict $O(d)$ mỗi mẫu |
| Hyperparameters | không có (OLS); Ridge/Lasso có $\lambda$ (bài 07) |
| Preprocessing | scaling không bắt buộc cho OLS nhưng cần cho GD, regularization, so sánh hệ số |
| Interpretation | hệ số = thay đổi kỳ vọng của $y$ khi $x_j$ tăng 1 đơn vị, giữ các feature khác cố định |

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Góc nhìn xác suất: Linear Regression = MLE với nhiễu Gauss

Giả định dữ liệu được sinh ra bởi:

$$y^{(i)} = \mathbf w^\top\mathbf x^{(i)} + b + \varepsilon^{(i)}, \qquad \varepsilon^{(i)} \overset{iid}{\sim} \mathcal N(0, \sigma^2)$$

tức $y \mid \mathbf x \sim \mathcal N(\mathbf w^\top\mathbf x + b,\ \sigma^2)$. Như đã chứng minh ở chương 02 bài 07, cực đại likelihood ⇔ cực tiểu tổng bình phương sai số. Góc nhìn này cho ta thêm: ước lượng $\sigma^2$, **khoảng tin cậy** cho hệ số và **khoảng dự đoán** cho $y$ mới.

In [ ]:
n = 200
x = rng.uniform(0, 10, n)
y = 3.0 + 1.5 * x + rng.normal(0, 2.0, n)
X = np.c_[np.ones(n), x]
w = np.linalg.lstsq(X, y, rcond=None)[0]
resid = y - X @ w
sigma2_hat = resid @ resid / (n - X.shape[1])            # ước lượng không chệch của σ²
cov_w = sigma2_hat * np.linalg.inv(X.T @ X)              # hiệp phương sai của ước lượng hệ số
se = np.sqrt(np.diag(cov_w))
t = stats.t.ppf(0.975, n - 2)
for name, est, s in zip(["b (intercept)", "w (slope)"], w, se):
    print(f"{name:14s} = {est:.3f}  CI95% = [{est - t * s:.3f}, {est + t * s:.3f}]")
print(f"σ ước lượng = {np.sqrt(sigma2_hat):.3f} (thật: 2.0)")

In [ ]:
grid = np.linspace(0, 10, 100)
Xg = np.c_[np.ones_like(grid), grid]
mean_pred = Xg @ w
se_mean = np.sqrt(np.einsum("ij,jk,ik->i", Xg, cov_w, Xg))                   # bất định của đường hồi quy
se_pred = np.sqrt(se_mean**2 + sigma2_hat)                                     # + nhiễu của từng điểm
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, s=10, alpha=0.5)
ax.plot(grid, mean_pred, "r", lw=2, label="đường hồi quy")
ax.fill_between(grid, mean_pred - t * se_mean, mean_pred + t * se_mean, alpha=0.4, label="CI 95% của trung bình")
ax.fill_between(grid, mean_pred - t * se_pred, mean_pred + t * se_pred, alpha=0.15, label="khoảng dự đoán 95%")
ax.legend()
ax.set_title("Bất định của đường hồi quy vs bất định của một dự đoán mới")
plt.show()

Khoảng dự đoán rộng hơn nhiều so với khoảng tin cậy của đường: ngay cả khi biết **chính xác** đường thật, từng điểm mới vẫn lệch ±2σ do nhiễu. Chương 37 (uncertainty, conformal prediction) mở rộng ý tưởng này cho mọi model.

## 2. Các giả định của Linear Regression

| Giả định | Ý nghĩa | Vi phạm thì sao? | Kiểm tra (bài 08) |
|---|---|---|---|
| **Tuyến tính** | $\mathbb E[y \mid \mathbf x]$ là hàm tuyến tính của $\mathbf x$ | dự đoán sai có hệ thống | residual vs fitted có hình cong |
| **Độc lập** | các sai số không phụ thuộc nhau | CI/p-value sai (quá lạc quan) | dữ liệu thời gian, nhóm |
| **Phương sai không đổi** (homoscedasticity) | $\operatorname{Var}(\varepsilon)$ như nhau mọi nơi | CI sai; model bị chi phối bởi vùng nhiễu lớn | residual "loe" hình phễu |
| **Nhiễu chuẩn** | $\varepsilon \sim \mathcal N$ | ảnh hưởng CI với mẫu nhỏ; outlier kéo lệch | Q-Q plot |
| **Không đa cộng tuyến mạnh** | feature không gần như phụ thuộc tuyến tính | hệ số không ổn định, khó diễn giải | VIF, condition number |

> Cho mục đích **dự đoán**, vi phạm nhẹ các giả định 3–4 thường chấp nhận được. Cho mục đích **suy luận** (diễn giải hệ số, p-value), các giả định rất quan trọng.

## 3. From scratch → scikit-learn

Class tham chiếu `LinearRegressionScratch` (file `src/ml_course/algorithms/linear_models.py`) theo đúng API sklearn, hỗ trợ cả nghiệm đóng và gradient descent:

In [ ]:
import inspect

from ml_course.algorithms import LinearRegressionScratch

print(inspect.getsource(LinearRegressionScratch))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../../data/sample/house_prices.csv")
features = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
X_train, X_test, y_train, y_test = train_test_split(df[features].to_numpy(float), df["price_billion_vnd"].to_numpy(),
                                                    test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)

ours_ne = LinearRegressionScratch(method="normal").fit(Xtr, y_train)
ours_gd = LinearRegressionScratch(method="gd", lr=0.1, n_iters=2000).fit(Xtr, y_train)
sk = LinearRegression().fit(Xtr, y_train)
print("scratch (normal):", ours_ne.coef_.round(4), round(ours_ne.intercept_, 4))
print("scratch (GD)    :", ours_gd.coef_.round(4), round(ours_gd.intercept_, 4))
print("scikit-learn    :", sk.coef_.round(4), round(sk.intercept_, 4))

## 4. Diễn giải hệ số — đúng và sai

Fit trên dữ liệu **gốc** (chưa chuẩn hoá) để hệ số có đơn vị thực:

In [ ]:
raw = LinearRegression().fit(X_train, y_train)
pd.DataFrame({"hệ số (đơn vị gốc)": raw.coef_, "hệ số (đã chuẩn hoá)": sk.coef_}, index=features).round(4)

- **Đơn vị gốc:** hệ số `area_m2` ≈ 0.057 nghĩa là: thêm **1 m²** (giữ số phòng, tuổi, khoảng cách **không đổi**) thì giá kỳ vọng tăng khoảng 0.057 tỷ (≈ 57 triệu). Không so sánh độ lớn hệ số giữa các feature khác đơn vị!
- **Đã chuẩn hoá:** thay đổi khi feature tăng **1 độ lệch chuẩn** → so sánh tương đối được mức ảnh hưởng.
- ⚠️ **"Giữ các feature khác không đổi"** — hệ số là ảnh hưởng **có điều kiện**, khác hẳn tương quan đơn. Ví dụ dưới đây: `bedrooms` tương quan khá mạnh với giá (≈ 0.6), nhưng hệ số của nó trong model đa biến gần như **bằng 0** — vì khi đã biết diện tích, số phòng ngủ **không mang thêm thông tin** về giá (dữ liệu này được sinh ra với giá phụ thuộc diện tích chứ không phụ thuộc số phòng). Tương quan đơn của `bedrooms` chỉ là "cái bóng" của diện tích. Trong dữ liệu thật, hệ số như vậy thậm chí có thể **âm** (cùng diện tích, nhiều phòng hơn = phòng nhỏ hơn).
- ⚠️ Hệ số **không** phải quan hệ **nhân quả** (chương 38).

In [ ]:
print("Tương quan đơn giữa bedrooms và giá:", round(np.corrcoef(X_train[:, 1], y_train)[0, 1], 3))
print("Hệ số của bedrooms trong model đa biến:", round(raw.coef_[1], 4))

### Bảng thống kê đầy đủ với statsmodels (tuỳ chọn)

`statsmodels` cung cấp bảng hồi quy kiểu thống kê: sai số chuẩn, t-stat, p-value, CI cho từng hệ số, R² điều chỉnh...

In [ ]:
try:
    import statsmodels.api as sm

    ols = sm.OLS(y_train, sm.add_constant(pd.DataFrame(X_train, columns=features))).fit()
    print(ols.summary().tables[1])
except ImportError:
    print("Chưa cài statsmodels (pip install statsmodels) — bỏ qua phần tuỳ chọn.")

## 5. Điểm mạnh, điểm yếu, khi nào dùng

| ✅ Điểm mạnh | ❌ Điểm yếu |
|---|---|
| Rất nhanh, nghiệm đóng, không có hyperparameter | Chỉ biểu diễn quan hệ tuyến tính (cần tự tạo feature phi tuyến) |
| Dễ diễn giải, có lý thuyết suy luận thống kê đầy đủ | Nhạy với outlier (bình phương sai số) |
| Ít dữ liệu vẫn chạy được, ít overfit khi $n \gg d$ | Không ổn định khi đa cộng tuyến mạnh |
| Baseline tuyệt vời cho mọi bài toán hồi quy | Ngoại suy tuyến tính vô hạn — có thể vô lý (giá âm) |

**Dùng khi:** cần baseline, cần diễn giải, quan hệ gần tuyến tính, dữ liệu ít. **Thận trọng khi:** nhiều outlier, quan hệ phi tuyến mạnh, tương tác phức tạp → cân nhắc Ridge/Lasso (bài 07), Huber, tree ensemble (chương 11–12).

## 🧠 Kiểm tra nhanh

1. Giả định nhiễu Gauss liên quan gì tới việc chọn MSE làm loss?
2. Vì sao khoảng dự đoán luôn rộng hơn khoảng tin cậy của đường hồi quy?
3. Hệ số của `bedrooms` âm — có phải thêm phòng ngủ làm nhà rẻ đi không? Giải thích.
4. Kể 3 giả định của Linear Regression và cách kiểm tra từng giả định.

## 📚 Tài liệu tham khảo

- James, Witten, Hastie, Tibshirani — *An Introduction to Statistical Learning* (ISLR), chương 3 (miễn phí)
- Gelman, Hill, Vehtari — *Regression and Other Stories* (2020)